### CRISP-DM Phase 4.2 - Modeling : Country-level correlation

In [ ]:
import pandas as pd
import ast
import numpy as np
from scipy.stats import spearmanr

In [ ]:
# Load the datasets
sensor = pd.read_csv('data/sensor.csv')
law = pd.read_csv('../law/data/law_classified.csv')

Distribution of EU countries

In [ ]:
def distribute_eu(df, members):
    eu_docs = df[df['Country'].isin(['EUR', 'EUE'])].copy()
    non_eu_docs = df[~df['Country'].isin(['EUR', 'EUE'])].copy()
    
    expanded = []
    for _, row in eu_docs.iterrows():
        for member in members:
            new_row = row.copy()
            new_row['Country'] = member
            new_row['eu_distributed'] = True
            expanded.append(new_row)
    
    eu_expanded = pd.DataFrame(expanded)
    result = pd.concat([non_eu_docs, eu_expanded], ignore_index=True)

    return result

In [ ]:
EU_MEMBERS = ['AUT', 'BEL', 'BGR', 'HRV', 'CYP', 'CZE', 'DNK', 'EST', 'FIN', 'FRA',
              'DEU', 'GRC', 'HUN', 'IRL', 'ITA', 'LVA', 'LTU', 'LUX', 'MLT', 'NLD',
              'POL', 'PRT', 'ROU', 'SVK', 'SVN', 'ESP', 'SWE']

law = distribute_eu(law, EU_MEMBERS)

Compute relevant metrics

In [ ]:
## Legislative coverage
START = 2000
END = 2026
YEARS = list(range(START, END + 1))
HAZARDS = ['flood', 'drought', 'temperature_extremes', 'sea_level_rise', 'storm', 'wildfire',
          'melting', 'erosion', 'other', 'none']

law['Hazard'] = law['Hazard'].apply(lambda x: x if isinstance(x, list) else ast.literal_eval(x))
law_exploded = law.explode('Hazard').copy()

legislative_coverage = []

for country, df in law_exploded.groupby('Country', sort=True):
    for hazard in HAZARDS:
        subset = df[(df['Hazard'] == hazard) & (df['Year'] >= START) & (df['Year'] <= END)].sort_values('Year').copy()

        year_df = pd.DataFrame({'Year': YEARS})
        year_df['Country'] = country
        year_df['Hazard'] = hazard

        if subset.empty:
            year_df['Count'] = 0
        else:
            yearly_counts = subset.groupby('Year').size().reset_index(name='new_laws')
            year_df = year_df.merge(yearly_counts, on='Year', how='left')
            year_df['new_laws'] = year_df['new_laws'].fillna(0)
            year_df['Count'] = year_df['new_laws'].cumsum()

            # If last law appears before 2026
            last_year_with_law = subset['Year'].max()
            if last_year_with_law < END:
                final_count = year_df.loc[year_df['Year'] == last_year_with_law, 'Count'].values[0]
                year_df.loc[year_df['Year'] > last_year_with_law, 'Count'] = final_count

        legislative_coverage.append(year_df[['Country', 'Year', 'Hazard', 'Count']])

legislative_coverage = pd.concat(legislative_coverage, ignore_index=True)
legislative_coverage.to_csv(f'../law/outputs/legislative_coverage_country.csv', index=False)

In [ ]:
## Hazard intensity 
VARIABLES = ['2m_temperature', 'Instantaneous_wind_gust', 'Sea_level_anomaly', 
             'Snowmelt', 'SPEI', 'Total_precipitation']
BASELINE_END = 1999

hazard_intensity = sensor.copy()

for var in VARIABLES:
    if var == 'Sea_level_anomaly' or var == 'Snowmelt':
        # SPEI (reference period 1991–2020) and SLA (reference period 1993-2012) already usable as is
        continue
    
    baseline = sensor[sensor['Year'] <= BASELINE_END].groupby('Country')[var].agg(mean='mean', std='std').reset_index()
    baseline.columns = ['Country', f'{var}_mean', f'{var}_std'] 
    hazard_intensity = hazard_intensity.merge(baseline, on='Country', how='left')
    
    hazard_intensity[var] = ((hazard_intensity[var] - hazard_intensity[f'{var}_mean']) / 
        hazard_intensity[f'{var}_std'])
    
    hazard_intensity = hazard_intensity.drop(columns=[f'{var}_mean', f'{var}_std'])
    
hazard_intensity = hazard_intensity[(hazard_intensity['Year'] >= 2000) & (hazard_intensity['Year'] <= 2026)].copy()
hazard_intensity = hazard_intensity[['Country', 'Year'] + [v for v in VARIABLES]].copy()
hazard_intensity = hazard_intensity.groupby(['Country', 'Year'], as_index=False).mean()
hazard_intensity['Snowmelt'] = hazard_intensity['Snowmelt'].replace([np.inf], np.nan)
hazard_intensity.to_csv(f'outputs/hazard_intensity_country.csv', index=False)

Correlation

In [ ]:
hazard_variable_dict = {'flood': 'Total_precipitation', 'drought': 'SPEI', 
                        'temperature_extremes': '2m_temperature', 'sea_level_rise': 'Sea_level_anomaly', 
                        'storm': 'Instantaneous_wind_gust', 'melting': 'Snowmelt'}

sensor_countries = set(hazard_intensity['Country'].dropna().unique())
law_countries = set(legislative_coverage['Country'].dropna().unique())
common_countries = sensor_countries & law_countries

correlation_results = []

for hazard, variable in hazard_variable_dict.items():
    for country in common_countries:
        country_intensity = hazard_intensity[hazard_intensity['Country'] == country]
        country_coverage = legislative_coverage[(legislative_coverage['Country'] == country) & (legislative_coverage["Hazard"] == hazard)]

        data = pd.merge(country_intensity[['Year', variable]], country_coverage[['Year', 'Count']], 
                        on='Year', how='inner').dropna()

        if data[variable].nunique() == 1 or data["Count"].nunique() == 1:  
            continue          
        
        try:
            rho, p = spearmanr(data[variable], data['Count'])
            correlation_results.append({'Hazard': hazard, 'Country': country,
                                        'Rho': rho, 'p_value': p}) 
        except Exception as e:
            continue

correlation_results = pd.DataFrame(correlation_results)
correlation_results.to_csv(f'outputs/correlation_results_country.csv', index=False)